# Legal-BERT Fine-Tuning on Google Colab

This notebook trains Legal-BERT for COBS classification using Colab's free GPU.

**Steps:**
1. Upload 3 CSV files (`train.csv`, `val.csv`, `test.csv`) and `label_map.json` from `data/processed/`
2. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
3. Run all cells

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
# Cell 2 — Upload data files
from google.colab import files
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('models/transformers/legal-bert', exist_ok=True)

print('Upload these 4 files from data/processed/ folder:')
print('  1. train.csv')
print('  2. val.csv')
print('  3. test.csv')
print('  4. label_map.json')

uploaded = files.upload()

for fname in uploaded:
    dest = f'data/processed/{fname}'
    os.rename(fname, dest)
    print(f'  → {dest}')

In [ ]:
# Cell 3 — Verify GPU and imports
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import pandas as pd, numpy as np, json
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns

In [ ]:
# Cell 4 — Config
MODEL_NAME   = 'nlpaueb/legal-bert-base-uncased'
OUTPUT_DIR   = 'models/transformers/legal-bert'
MAX_LEN      = 512
BATCH_SIZE   = 16   # Colab GPU can handle 16
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.1
SEED         = 42

In [ ]:
# Cell 5 — Load data & label map
train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')
test_df  = pd.read_csv('data/processed/test.csv')

with open('data/processed/label_map.json') as f:
    lm = json.load(f)
ID2LABEL_FULL = {int(k): v for k, v in lm['id2label'].items()}

# Only keep labels actually present
present_labels = sorted(int(x) for x in
    set(train_df['label'].unique()) |
    set(val_df['label'].unique()) |
    set(test_df['label'].unique()))
NUM_LABELS = len(present_labels)
ID2LABEL = {i: ID2LABEL_FULL[i] for i in present_labels}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print(f'Labels found: {ID2LABEL} (NUM_LABELS={NUM_LABELS})')
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
# Cell 6 — Tokenise
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding=False)

train_ds = Dataset.from_pandas(train_df[['text','label']]).map(tokenise, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text','label']]).map(tokenise, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']]).map(tokenise, batched=True)
print('Tokenisation done.')

In [ ]:
# Cell 7 — Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f'Model loaded: {MODEL_NAME} → {NUM_LABELS} labels')

In [ ]:
# Cell 8 — Metrics (using sklearn — no downloads needed)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1  = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'f1_macro': f1, 'accuracy': acc}

In [ ]:
# Cell 9 — Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

In [ ]:
# Cell 10 — Evaluate on test set
preds_out = trainer.predict(test_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = test_df['label'].values

all_labels = sorted(set(y_true) | set(y_pred))
label_names = [ID2LABEL[i] for i in all_labels]

print('\n── Test Results ──')
print(classification_report(y_true, y_pred, labels=all_labels, target_names=label_names))

# Save report JSON
report = classification_report(y_true, y_pred, labels=all_labels, target_names=label_names, output_dict=True)
with open('data/processed/legal_bert_report.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
# Cell 11 — Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_title('Legal-BERT Confusion Matrix (Test Set)')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('data/processed/cm_legal_bert.png', dpi=150)
plt.show()

In [ ]:
# Cell 12 — Save model & download results
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved → {OUTPUT_DIR}')

# Download key outputs back to your local machine
from google.colab import files
files.download('data/processed/legal_bert_report.json')
files.download('data/processed/cm_legal_bert.png')